In [1]:
# ============================================================
# MODULE 4.1 — PDF LOGICAL SPLITTING
# Boundary Detection Pipeline
# ============================================================

# ── CELL 1: CÀI ĐẶT ─────────────────────────────────────────
!pip install pymupdf pdfplumber pdf2image pytesseract -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie -q

import fitz, pdfplumber, re, json
import pytesseract
from pdf2image import convert_from_path
from google.colab import drive

print("✅ Setup xong!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (4,871 kB/s)
Selec

In [2]:
# ── CELL 2: CONFIG ───────────────────────────────────────────
PDF_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged.pdf'

# Regex bắt Số văn bản (VD: 913/2025/QĐST-HC, 69/2009/NĐ-CP)
RE_SO_VAN_BAN = r'\d{1,3}/\d{4}/[A-ZĐQĐ][\w\-]+'

# Ground truth để test
GT_BOUNDARIES = {1, 23,29, 32, 35, 44} #
GT_SEPARATORS = {3, 29}

with fitz.open(PDF_PATH) as doc:
    TOTAL_PAGES = len(doc)

print(f"✅ Config xong | {PDF_PATH.split('/')[-1]} | {TOTAL_PAGES} trang")

✅ Config xong | document_mix.pdf | 27 trang


In [3]:
# ── CELL 3: EXTRACT TEXT & FEATURES ────────────────────────
import io
from PIL import Image

def get_text_hybrid_fast(pdf_path: str, page_idx: int) -> str:
    with fitz.open(pdf_path) as doc:
        page = doc[page_idx]
        text = page.get_text("text").strip()

        # Nếu không có text gốc (ảnh scan), chuyển sang OCR tốc độ cao
        if len(text) < 10:
            pix = page.get_pixmap(dpi=150, colorspace=fitz.csGRAY)
            img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, lang='vie').strip()

    return text


def extract_features(pdf_path: str, page_idx: int) -> dict:
    # ĐÃ SỬA LỖI: Gọi đúng hàm hybrid OCR
    text = get_text_hybrid_fast(pdf_path, page_idx)
    lines = text.split("\n") if text else []
    n = len(lines)

    # Gom 20% số dòng đầu tiên làm Header (tối thiểu 5 dòng) để định vị
    header = " ".join(l.strip() for l in lines[:max(5, int(n * 0.20))] if l.strip())
    header_up = header.upper()

    # 1. Tìm Quốc hiệu / Tiêu ngữ / Tên cơ quan đặc trưng TẠI HEADER
    has_quoc_hieu = "CỘNG HÒA XÃ HỘI" in header_up
    has_tieu_ngu = "HẠNH PHÚC" in header_up
    has_toa_an = "TÒA ÁN NHÂN DÂN" in header_up

    # 2. Bắt Số văn bản, nhưng CHỈ BẮT Ở HEADER để tránh dính trích dẫn
    so_vb_header = re.findall(RE_SO_VAN_BAN, header)

    return {
        "page_num": page_idx + 1,
        "char_count": len(text),
        "is_blank": len(text) < 30,
        "header": header,
        "has_quoc_hieu": has_quoc_hieu,
        "has_tieu_ngu": has_tieu_ngu,
        "has_toa_an": has_toa_an,
        "so_vb_header": so_vb_header,
        "text_full": text,
    }

# Chạy extract
print("⏳ Đang extract features (sử dụng Hybrid OCR)...\n")
print(f"{'Trang':<7}{'Ký tự':<8}{'Blank':<7}{'Quốc Hiệu':<12}{'Tòa Án':<10}{'Số VB (Header)'}")
print("─" * 75)

all_features = []
for i in range(TOTAL_PAGES):
    f = extract_features(PDF_PATH, i)
    all_features.append(f)

    so_vb_str = f["so_vb_header"][0] if f["so_vb_header"] else "—"
    qh_str = "✅" if f['has_quoc_hieu'] else "—"
    ta_str = "✅" if f['has_toa_an'] else "—"

    print(
        f"  {f['page_num']:<5}"
        f"{f['char_count']:<8}"
        f"{'🔴' if f['is_blank'] else '':<7}"
        f"{qh_str:<12}"
        f"{ta_str:<10}"
        f"{so_vb_str}"
    )

print("─" * 75)
print(f"✅ Extract xong {len(all_features)} trang!")

⏳ Đang extract features (sử dụng Hybrid OCR)...

Trang  Ký tự   Blank  Quốc Hiệu   Tòa Án    Số VB (Header)
───────────────────────────────────────────────────────────────────────────
  1    2016           ✅           ✅         01/2026/QĐ-PT
  2    1849           —           —         01/2026/TLPT-HC
  3    1663           —           ✅         01/2025/QĐST-HC
  4    1762           —           ✅         46/2025/QĐST-HC
  5    1006           —           —         —
  6    1956           ✅           ✅         61/2026/HS-PT
  7    2650           —           —         —
  8    2543           —           —         —
  9    2624           —           —         —
  10   2686           —           —         —
  11   2671           —           —         —
  12   2939           —           —         —
  13   2136           —           —         —
  14   908            —           —         326/2016/UBTVQHI4
  15   1767           —           ✅         75/2026/HS-PT
  16   1851           —         

In [4]:
# ── CELL 4: BOUNDARY DETECTION ───────────────────────────────
def is_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    # 1. Trang đầu tiên luôn là ranh giới
    if prev is None:
        return True, "first_page", 1.0

    # 2. Bỏ qua trang trắng
    if feat["is_blank"]:
        return False, "blank", 0.0

    signals = []
    conf = 0.0

    # 3. LUẬT MẠNH 1: Có Quốc hiệu + Tiêu ngữ ở phần đầu trang
    if feat["has_quoc_hieu"] and feat["has_tieu_ngu"]:
        signals.append("quoc_hieu_tieu_ngu_in_header")
        conf += 1.0

    # 4. LUẬT MẠNH 2: Có cụm "Tòa án" (hoặc cơ quan khác) và có Số văn bản ở Header
    elif feat["has_toa_an"] and feat["so_vb_header"]:
        signals.append(f"toa_an_va_so_vb_in_header:{feat['so_vb_header'][0]}")
        conf += 0.9

    return conf >= 0.8, " + ".join(signals) if signals else "continuation", conf

print("BOUNDARY DETECTION RESULTS")
print("=" * 95)
print(f"{'Trang':<7}{'Predict':<13}{'GT':<13}{'Match':<9}{'Conf':<7}{'Reason'}")
print("─" * 95)

predict_set = set()
for i, feat in enumerate(all_features):
    prev = all_features[i-1] if i > 0 else None
    bdry, reason, conf = is_boundary(feat, prev)
    pn = feat["page_num"]

    if bdry:
        predict_set.add(pn)

    gt_label   = "BOUNDARY" if pn in GT_BOUNDARIES else ("SEPARATOR" if pn in GT_SEPARATORS else "continuation")
    pred_label = "BOUNDARY" if bdry else ("separator" if feat["is_blank"] else "continuation")
    match      = ("✅ HIT" if bdry else "❌ MISS") if pn in GT_BOUNDARIES else \
                 ("⚠️ FP"  if bdry else "") if pn not in GT_SEPARATORS else ""

    if match or pn in GT_BOUNDARIES | GT_SEPARATORS:
        print(f"  {pn:<5}{pred_label:<13}{gt_label:<13}{match:<9}{conf:<7}{reason[:55]}")

print("=" * 95)

BOUNDARY DETECTION RESULTS
Trang  Predict      GT           Match    Conf   Reason
───────────────────────────────────────────────────────────────────────────────────────────────
  1    BOUNDARY     BOUNDARY     ✅ HIT    1.0    first_page
  3    BOUNDARY     SEPARATOR             0.9    toa_an_va_so_vb_in_header:01/2025/QĐST-HC
  4    BOUNDARY     continuation ⚠️ FP    0.9    toa_an_va_so_vb_in_header:46/2025/QĐST-HC
  6    BOUNDARY     continuation ⚠️ FP    1.0    quoc_hieu_tieu_ngu_in_header
  15   BOUNDARY     continuation ⚠️ FP    0.9    toa_an_va_so_vb_in_header:75/2026/HS-PT
  22   BOUNDARY     continuation ⚠️ FP    1.0    quoc_hieu_tieu_ngu_in_header
  23   continuation BOUNDARY     ❌ MISS   0.0    continuation


In [5]:
# ── CELL 5: METRICS ──────────────────────────────────────────
TP = len(predict_set & GT_BOUNDARIES)
FP = len(predict_set - GT_BOUNDARIES)
FN = len(GT_BOUNDARIES - predict_set)

p  = TP / (TP + FP) if (TP + FP) > 0 else 0
r  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*p*r / (p+r)  if (p + r)   > 0 else 0

print("📊 KẾT QUẢ")
print("=" * 50)
print(f"  GT predict    : {sorted(GT_BOUNDARIES)}")
print(f"  Predict       : {sorted(predict_set)}")
print(f"  TP / FP / FN  : {TP} / {FP} / {FN}")
print(f"  Precision     : {p:.3f}")
print(f"  Recall        : {r:.3f}")
print(f"  F1 Score      : {f1:.3f}  {'✅ Đạt' if f1 >= 0.90 else '⚠️ Chưa đạt 0.90'}")
print("=" * 50)

for pn in sorted(GT_BOUNDARIES - predict_set):
    f = all_features[pn-1]
    print(f"\n❌ Miss trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

for pn in sorted(predict_set - GT_BOUNDARIES):
    f = all_features[pn-1]
    print(f"\n⚠️  FP  trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

📊 KẾT QUẢ
  GT predict    : [1, 23, 29, 32, 35, 44]
  Predict       : [1, 3, 4, 6, 15, 22]
  TP / FP / FN  : 1 / 5 / 5
  Precision     : 0.167
  Recall        : 0.167
  F1 Score      : 0.167  ⚠️ Chưa đạt 0.90

❌ Miss trang 23: Q.Hiệu=False | T.Án=False | Số VB=[]
   Header: Theo các tài liệu có trong hồ sơ vụ án và diễn biến tại phiên tòa, nội dung vụ án được tóm tắt như sau: Vào khoảng II gi


IndexError: list index out of range

In [ ]:
# ── CELL 6: FIX GROUND TRUTH + KIỂM TRA HEADER ──────────────
# Trang 29 đang bị conflict (vừa BOUNDARY vừa SEPARATOR) → fix lại
GT_BOUNDARIES = {1, 23, 29, 32, 35, 44}  # trang đầu mỗi tài liệu
GT_SEPARATORS = {3}                        # chỉ trang trắng thật sự

print("KIỂM TRA HEADER TỪNG TRANG ĐỂ XÁC NHẬN GROUND TRUTH")
print("=" * 70)

# In header tất cả trang để xác nhận
for f in all_features:
    pn     = f["page_num"]
    tag    = " ← BOUNDARY" if pn in GT_BOUNDARIES else \
             (" ← SEPARATOR" if pn in GT_SEPARATORS else "")
    blank  = " 🔴 TRẮNG" if f["is_blank"] else ""
    print(f"\n[Trang {pn:02d}]{blank}{tag}")
    if not f["is_blank"]:
        print(f"  {f['header'][:120]}")

print("\n" + "=" * 70)
print(f"✅ GT_BOUNDARIES : {sorted(GT_BOUNDARIES)}")
print(f"✅ GT_SEPARATORS : {sorted(GT_SEPARATORS)}")
print("⚠ Nếu header trang nào không khớp → sửa GT_BOUNDARIES ở cell này rồi chạy lại")

In [ ]:
# ── CELL 7 (CẬP NHẬT): STAGE 3 — SEGMENT CLASSIFICATION ──────

# ── Keyword đặc trưng — ưu tiên EXCLUSIVE keyword ────────────
# Mỗi keyword chỉ nên xuất hiện ở đúng 1 loại tài liệu
SEGMENT_KEYWORDS = {
    "Bản án hành chính": [
        "hành chính",           # loại vụ án
        "người khởi kiện",      # exclusive: chỉ có ở HC
        "người bị kiện",        # exclusive
        "quyết định hành chính",
        "khiếu kiện",           # exclusive
        "hủy quyết định",
    ],
    "Bản án hình sự": [
        "hình sự",
        "bị cáo",               # exclusive: chỉ có ở HS
        "viện kiểm sát",        # exclusive
        "tội danh",             # exclusive
        "phạt tù",              # exclusive
        "tuyên phạt",
    ],
    "Bản án dân sự": [
        "dân sự",
        "nguyên đơn",           # exclusive: chỉ có ở DS
        "bị đơn",               # exclusive
        "tranh chấp",
        "hợp đồng",
        "thừa kế",
    ],
    "Quyết định": [
        "quyết định:",          # dòng tiêu đề riêng (có dấu :)
        "căn cứ vào điều",      # exclusive: chỉ QĐ mới dùng
        "đình chỉ giải quyết",  # exclusive
        "tạm ứng án phí",       # exclusive
        "nơi nhận:",            # exclusive: phần cuối QĐ
    ],
}

# Keyword MẠNH — nếu xuất hiện thì cộng thêm điểm lớn
STRONG_TYPE_KW = {
    "Bản án hành chính": ["bản án hành chính", "vụ án hành chính"],
    "Bản án hình sự":    ["bản án hình sự", "vụ án hình sự", "hsst", "hspt"],
    "Bản án dân sự":     ["bản án dân sự", "vụ án dân sự", "ds-st", "ds-pt"],
    "Quyết định":        ["quyết định đình chỉ", "quyết định hủy", "quyết định công nhận"],
}

# Ground truth — dựa trên header thực tế từ cell 6
GT_SEGMENT_TYPES = {
    1:  "Bản án hành chính",    # "Bản án số: 42" — TÒA ÁN CẤP CAO TPHCM
    23: "Bản án dân sự",        # "Bản án số: 03/2017/DS-ST"
    29: "Quyết định",           # header rỗng — xem lại nếu sai
    32: "Quyết định",           # "01/2026/QĐ-PT"
    35: "Bản án hình sự",       # "Bản án số: 04/2017/HSST"
    44: "Bản án hình sự",       # header rỗng — xem lại nếu sai
}


def classify_segment(pages: list) -> tuple[str, float, dict]:
    """
    Phân loại loại tài liệu cho segment.
    Chiến lược:
      1. Strong keyword check → nếu có thì quyết định ngay
      2. Exclusive keyword voting trên full text
      3. Header-only fallback cho segment ngắn (≤ 3 trang)
    """
    full_text  = " ".join(f["text_full"] for f in pages)
    full_lower = full_text.lower()

    # Header của trang đầu segment — quan trọng nhất
    header_lower = pages[0]["header"].lower() if pages else ""

    scores = {label: 0.0 for label in SEGMENT_KEYWORDS}

    # ── Bước 1: Strong keyword — quyết định ngay ────────────
    for label, kws in STRONG_TYPE_KW.items():
        for kw in kws:
            if kw in full_lower or kw in header_lower:
                scores[label] = scores.get(label, 0) + 2.0   # trọng số cao

    # ── Bước 2: Exclusive keyword voting ────────────────────
    for label, kws in SEGMENT_KEYWORDS.items():
        hits = sum(1 for kw in kws if kw in full_lower)
        scores[label] += hits / len(kws)

    # ── Bước 3: Header boost — trang đầu quan trọng hơn ────
    header_scores = {label: 0.0 for label in SEGMENT_KEYWORDS}
    for label, kws in SEGMENT_KEYWORDS.items():
        header_hits = sum(1 for kw in kws if kw in header_lower)
        header_scores[label] = header_hits / len(kws)
    # Header có trọng số x1.5
    for label in scores:
        scores[label] += header_scores.get(label, 0) * 1.5

    # ── Tính confidence ──────────────────────────────────────
    best_label = max(scores, key=scores.get)
    total      = sum(scores.values())
    confidence = round(scores[best_label] / total, 3) if total > 0 else 0.0

    # Nếu tất cả scores đều = 0 (segment quá ngắn / text rỗng)
    if total == 0:
        best_label = "Không xác định"
        confidence = 0.0

    return best_label, confidence, {k: round(v, 3) for k, v in scores.items()}


# ── Chạy classify ────────────────────────────────────────────
# Dùng lại segments từ cell trước (group_segments đã chạy)
print("SEGMENT CLASSIFICATION RESULTS (CẢI THIỆN)")
print("=" * 100)
print(f"{'Seg':<5}{'Trang':<12}{'Trang':<9}{'Predict':<25}{'Conf':<7}{'Match':<7}{'GT Label'}")
print("─" * 100)

correct = 0
results = []   # ghi đè results cũ để cell 8 dùng lại

for i, pages in enumerate(segments):
    page_start = pages[0]["page_num"]
    page_end   = pages[-1]["page_num"]
    n_pages    = len(pages)

    label, conf, scores = classify_segment(pages)
    gt_label = GT_SEGMENT_TYPES.get(page_start, "?")

    is_correct = label == gt_label
    if is_correct:
        correct += 1
    match = "✅" if is_correct else "❌"

    results.append({
        "segment"    : i + 1,
        "page_start" : page_start,
        "page_end"   : page_end,
        "n_pages"    : n_pages,
        "type"       : label,
        "confidence" : conf,
        "gt_type"    : gt_label,
        "match"      : is_correct,
        "scores"     : scores,
    })

    print(f"  {i+1:<4}{str(page_start)+'-'+str(page_end):<12}{n_pages:<9}{label:<25}{conf:<7}{match}  {gt_label}")

print("=" * 100)
acc = correct / len(segments) if segments else 0
print(f"\n📊 Classification Accuracy: {correct}/{len(segments)} = {acc:.3f}  "
      f"{'✅ Đạt target 0.93' if acc >= 0.93 else '⚠️ Chưa đạt 0.93'}")

# Chi tiết segment sai
for r in results:
    if not r["match"]:
        print(f"\n❌ Segment {r['segment']} (trang {r['page_start']}–{r['page_end']}):")
        print(f"   Predict : {r['type']} (conf={r['confidence']:.3f})")
        print(f"   GT      : {r['gt_type']}")
        print(f"   Scores  : {r['scores']}")
        print(f"   Header  : {segments[r['segment']-1][0]['header'][:100]}")
        print(f"   → Xem header trên, bổ sung keyword đặc trưng vào SEGMENT_KEYWORDS")

# Kiểm tra trang 34 bị mất
all_covered = set()
for pages in segments:
    for f in pages:
        all_covered.add(f["page_num"])
missing = set(range(1, TOTAL_PAGES+1)) - all_covered - GT_SEPARATORS
if missing:
    print(f"\n⚠️  Trang không thuộc segment nào: {sorted(missing)}")
    for pn in sorted(missing):
        f = all_features[pn-1]
        print(f"   Trang {pn}: blank={f['is_blank']} | chars={f['char_count']} | header={f['header'][:60]}")

In [ ]:
# ── CELL 8: STAGE 4 — PDF SPLIT + JSON OUTPUT ────────────────
import uuid, time
from datetime import datetime
from pathlib import Path

OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def split_pdf(pdf_path: str, segments: list, results: list, output_dir: str) -> dict:
    """
    Tách PDF gốc thành các file con theo segment.
    Xuất JSON result đúng contract spec.
    """
    src        = fitz.open(pdf_path)
    doc_id     = str(uuid.uuid4())[:8]
    start_time = time.time()
    sub_docs   = []
    warnings   = []

    for r in results:
        seg_idx    = r["segment"] - 1
        page_start = r["page_start"] - 1   # fitz dùng 0-indexed
        page_end   = r["page_end"] - 1

        # Tên file output
        label_safe = r["type"].replace(" ", "_")
        filename   = f"sub_{r['segment']:03d}_{label_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_path   = f"{output_dir}/{filename}"

        # Tách và lưu file PDF con
        doc_out = fitz.open()
        doc_out.insert_pdf(src, from_page=page_start, to_page=page_end)
        doc_out.save(out_path)
        doc_out.close()

        sub_doc = {
            "sub_doc_id" : f"sub_{r['segment']:03d}",
            "type"       : r["type"],
            "page_start" : r["page_start"],
            "page_end"   : r["page_end"],
            "n_pages"    : r["n_pages"],
            "confidence" : r["confidence"],
            "output_file": filename,
        }
        sub_docs.append(sub_doc)

        # Warning nếu confidence thấp
        if r["confidence"] < 0.6:
            warnings.append({
                "sub_doc_id": f"sub_{r['segment']:03d}",
                "code"      : "LOW_CONFIDENCE_CLASSIFICATION",
                "message"   : f"Segment trang {r['page_start']}–{r['page_end']} có confidence thấp "
                              f"({r['confidence']:.2f}), đề nghị QA review.",
            })

    elapsed_ms = int((time.time() - start_time) * 1000)
    src.close()

    result_json = {
        "document_id"      : doc_id,
        "source_file"      : Path(pdf_path).name,
        "total_pages"      : TOTAL_PAGES,
        "total_segments"   : len(sub_docs),
        "processing_time_ms": elapsed_ms,
        "latency_per_page_ms": round(elapsed_ms / TOTAL_PAGES, 1),
        "created_at"       : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "sub_documents"    : sub_docs,
        "warnings"         : warnings,
    }

    # Lưu JSON
    json_path = f"{output_dir}/result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result_json, f, ensure_ascii=False, indent=2)

    return result_json, json_path


# ── Chạy split ───────────────────────────────────────────────
print("⏳ Đang tách PDF...")
result_json, json_path = split_pdf(PDF_PATH, segments, results, OUTPUT_DIR)

# In kết quả
print("\nPDF SPLIT RESULTS")
print("=" * 70)
for sub in result_json["sub_documents"]:
    print(f"  [{sub['sub_doc_id']}] Trang {sub['page_start']:>2}–{sub['page_end']:>2} "
          f"({sub['n_pages']:>2} trang) | {sub['type']:<28} | conf={sub['confidence']:.2f}")
    print(f"          └─ {sub['output_file']}")

print("=" * 70)
print(f"\n📄 JSON output : {json_path}")
print(f"⏱ Thời gian   : {result_json['processing_time_ms']} ms "
      f"({result_json['latency_per_page_ms']} ms/trang)")
print(f"⚠ Warnings    : {len(result_json['warnings'])} segment cần QA review")

if result_json["warnings"]:
    for w in result_json["warnings"]:
        print(f"   → {w['message']}")

# In JSON mẫu để kiểm tra format
print("\nJSON OUTPUT (xem trước):")
print(json.dumps(result_json, ensure_ascii=False, indent=2)[:800] + "\n...")

# ── Verify: đọc lại từng file PDF con, in số trang + header ──
print("\nVERIFY OUTPUT FILES")
print("─" * 70)
for sub in result_json["sub_documents"]:
    path = f"{OUTPUT_DIR}/{sub['output_file']}"
    with fitz.open(path) as doc:
        n     = len(doc)
        head  = doc[0].get_text("text").strip()[:80].replace("\n", " ")
    status = "✅" if n == sub["n_pages"] else "❌ SỐ TRANG LỆCH"
    print(f"  {status} {sub['output_file']}")
    print(f"     Số trang: {n} | Header: {head}")